# Data eksternal v1 (VALID saja, test tidak dinilai)

Resep demo persis (M2M100-418M, 15 epoch, Krama Alus x3, multi-task, rerank lambda 2, seed 42). Yang berbeda hanya data latih tambahan.

| Lengan | Data tambahan |
|---|---|
| `base` | tidak ada; greedy chrF++ per epoch harus identik dengan f100 learning_curve_v2 |
| `ngokokrama` | NgokoKrama (Ngoko, Krama, Krama Alus) |
| `tanpa_ngokokrama` | NusaX + NusaTranslation (Ngoko) |
| `semua` | ketiganya |

Data tambahan 1x (tidak di-oversampling). Langkah training mengikuti jumlah data, jadi efek data dan langkah belum terpisah.
Aturan baca (dikunci sebelum push): lengan menang jika skor valid tertinggi DAN chrF++ Krama Alus tidak turun > 1,0 dari base.
1 seed; klaim perbaikan Krama Alus tetap butuh audit makna. Data eksternal diunduh saat run dan tidak disimpan di output
(NgokoKrama tidak berlisensi).

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sacrebleu", "sentencepiece", "openpyxl", "scikit-learn"], check=False)

import glob, json, time, random, re, os, hashlib, gc, shutil
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import numpy as np, pandas as pd, torch, sacrebleu, transformers
import torch.nn.functional as F
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)

REPO, SL, TL = "facebook/m2m100_418M", "id", "jv"
TRAIN_SEED = 42
EPOCHS, OS, BS, LR, WARMUP, JUDGE_SEED, MAXLEN, CTRL_W = 15, 3, 16, 5e-5, 0.1, 42, 64, 30.0
NBEST, LAM = 8, 2.0
OUT = "/kaggle/working"
dev = "cuda"
LEVELS = ["ngoko", "krama", "krama_alus"]
LID = {l: i for i, l in enumerate(LEVELS)}
TAG = {l: f"<{l}>" for l in LEVELS}
INIT_WORDS = {"ngoko": "kowe aku mangan lunga turu", "krama": "sampeyan kula nedha kesah tilem",
              "krama_alus": "panjenenganipun dalem dhahar tindak sare"}
T0 = time.time()
log = lambda *a: print(f"[{time.time()-T0:6.0f}s]", *a, flush=True)
sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()


def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


find = lambda n: sorted(p for p in glob.glob(f"/kaggle/input/**/{n}.csv", recursive=True) if "/output" not in p)[0]
tr_raw = pd.read_csv(find("train"))
def _load3(n):
    d = pd.read_csv(find(n))
    d = d[d["level"].isin(LEVELS)].copy()
    d["label"] = d["level"].map(LID)
    return d.reset_index(drop=True)
tr, va, te = (_load3(n) for n in ["train", "valid", "test"])
assert (len(tr), len(va), len(te)) == (1395, 175, 176), (len(tr), len(va), len(te))
RR_DIR = os.path.dirname(glob.glob("/kaggle/input/**/model/reranker/config.json", recursive=True)[0])
log("data", len(tr), len(va), len(te), torch.cuda.get_device_name(0), "| reranker", RR_DIR)
tag_src = lambda ind, level: f"{TAG[level]} {ind}"
def bleu(h, r): return sacrebleu.corpus_bleu(h, [r]).score
def chrf(h, r): return sacrebleu.corpus_chrf(h, [r], word_order=2).score
_rr_w = glob.glob(f"{RR_DIR}/*.safetensors") + glob.glob(f"{RR_DIR}/*.bin")
MANIFEST = {"protokol": "data eksternal v1", "train_seed": TRAIN_SEED,
            "hparams": {"epochs": EPOCHS, "os": OS, "bs": BS, "lr": LR, "warmup": WARMUP, "maxlen": MAXLEN,
                        "ctrl_w": CTRL_W, "nbest": NBEST, "lambda": LAM, "judge_seed": JUDGE_SEED},
            "versions": {"python": sys.version.split()[0], "torch": torch.__version__,
                         "transformers": transformers.__version__, "sacrebleu": sacrebleu.__version__},
            "gpu": torch.cuda.get_device_name(0),
            "inputs": {n: sha(find(n)) for n in ["train", "valid", "test"]},
            "reranker_sha256": sha(_rr_w[0]) if _rr_w else None,
            "runs": {}}

In [ ]:
# ---------------------------------------------------- juri kernel (kode v2 persis) + reranker demo (dimuat)
def train_classifier(base, seed):
    ctok = AutoTokenizer.from_pretrained(base)
    cmod = AutoModelForSequenceClassification.from_pretrained(base, num_labels=len(LEVELS)).to(dev)
    seed_all(seed)
    opt = torch.optim.AdamW(cmod.parameters(), lr=5e-5)
    X, y = tr["javanese"].tolist(), tr["label"].tolist()
    for _ in range(4):
        cmod.train(); order = list(range(len(X))); random.shuffle(order)
        for i in range(0, len(order), 32):
            b = order[i:i + 32]
            enc = ctok([X[j] for j in b], return_tensors="pt", padding=True, truncation=True, max_length=MAXLEN).to(dev)
            loss = cmod(**enc, labels=torch.tensor([y[j] for j in b], device=dev)).loss
            opt.zero_grad(); loss.backward(); opt.step()
    cmod.eval()
    return ctok, cmod


def make_lp(ctok, cmod):
    def logprobs(sents):
        out = []
        for i in range(0, len(sents), 64):
            enc = ctok(sents[i:i + 64], return_tensors="pt", padding=True, truncation=True, max_length=MAXLEN).to(dev)
            with torch.no_grad():
                out.append(F.log_softmax(cmod(**enc).logits.float(), -1).cpu())
        return torch.cat(out).numpy()
    return logprobs


def clf_info(lp):
    info = {}
    for n, d in [("valid", va), ("test", te)]:
        p = lp(d["javanese"].tolist()).argmax(-1); y_ = d["label"].values
        info[f"acc_{n}"] = round(float((p == y_).mean()), 4)
        info[f"per_class_acc_{n}"] = {l: round(float((p[y_ == i] == i).mean()), 4) for i, l in enumerate(LEVELS)}
    return info


judge_lp = make_lp(*train_classifier("w11wo/javanese-distilbert-small", JUDGE_SEED))
classify = lambda s: judge_lp(s).argmax(-1).tolist()
rr_tok = AutoTokenizer.from_pretrained(RR_DIR)
rr_mod = AutoModelForSequenceClassification.from_pretrained(RR_DIR).to(dev).float().eval()
rr_lp = make_lp(rr_tok, rr_mod)
# v2: juri acc_valid 0.96 (KA 0.8148), test 0.9375; reranker acc_valid 0.9657, test 0.9261
MANIFEST["judge"] = clf_info(judge_lp); MANIFEST["reranker"] = clf_info(rr_lp)
log("juri", MANIFEST["judge"]); log("reranker", MANIFEST["reranker"])

In [ ]:
SKRIP = r'''"""Gabungkan Unggah-Ungguh (train saja) + NgokoKrama + NusaTranslation + NusaX untuk 3 tingkat.

Jalankan dari root repo:  python scripts/gabung_data_eksternal.py
Input : data/external/{ngokokrama/NgokoKrama.xlsx, nusatranslation/jav-*.csv, nusax/train.csv} (di-gitignore)
Output: data/merged/train_eksternal_3lv.csv, data/merged/train_eksternal_3lv_laporan.json (di-gitignore)

- Valid/test Unggah-Ungguh tidak disentuh. Baris eksternal yang mirip valid/test dibuang.
- Label tingkat baris eksternal ditentukan classifier TF-IDF 4 tingkat yang dilatih di train Unggah-Ungguh.
  Sengaja bukan juri DistilBERT / reranker RoBERTa, supaya evaluasi tidak sirkular.
- "Krama" NgokoKrama hanya boleh menjadi krama atau krama_alus; "ngoko" hanya ngoko. Selain itu dibuang.
- NgokoKrama tidak berlisensi: pakai lokal saja, jangan disebarkan. Nonaktifkan dengan --tanpa-ngokokrama.
"""
import json
import re
import sys

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline, make_union

LEVELS = ["ngoko", "krama", "krama_alus"]
MIN_KATA, MAX_KATA = 3, 25
P_MIN = {"ngoko": 0.60, "krama": 0.60, "krama_alus": 0.85}  # KA < 0,85 sering keliru (audit sampel)
MIRIP = 0.80          # ambang cosine dengan valid/test
CAP_NUSATRANS = 300  # supaya ngoko tidak menenggelamkan tingkat lain
SEED = 42


def norm(s):
    s = str(s).replace("“", '"').replace("”", '"').replace("‘", "'").replace("’", "'")
    return re.sub(r"\s+", " ", s).strip()


def kunci(s):
    return re.sub(r"\W+", " ", str(s).lower()).strip()


def split(n):
    return pd.read_csv(f"data/unggah_ungguh/split/{n}.csv")


tr, va, te = split("train"), split("valid"), split("test")
clf = make_pipeline(
    make_union(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, min_df=2),
               TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
    LogisticRegression(C=20, max_iter=3000, class_weight="balanced"))
clf.fit(tr["javanese"], tr["level"])
lap = {"akurasi_pelabel_di_valid": {}}
pv, kv = clf.predict(va["javanese"]), clf.predict_proba(va["javanese"]).max(1)
for l in clf.classes_:
    yakin = (pv == l) & (kv >= P_MIN.get(l, 0.60))
    lap["akurasi_pelabel_di_valid"][l] = {
        "recall": round(float((pv[va.level == l] == l).mean()), 3),
        "presisi_saat_yakin": round(float((va.level[yakin] == l).mean()), 3), "n_yakin": int(yakin.sum())}

rows = []
if "--tanpa-ngokokrama" not in sys.argv:
    nk = pd.read_excel("data/external/ngokokrama/NgokoKrama.xlsx", sheet_name="All")
    for c, peran in [("javanese ngoko", "ngoko"), ("javanese krama", "krama")]:
        rows.append(pd.DataFrame({"indonesian": nk["indonesian"], "javanese": nk[c], "source": "ngokokrama", "peran": peran}))
for s in ["train", "valid", "test"]:
    d = pd.read_csv(f"data/external/nusatranslation/jav-{s}.csv")
    rows.append(pd.DataFrame({"indonesian": d["ind_text"], "javanese": d["tgt_text"], "source": "nusatranslation", "peran": "umum"}))
d = pd.read_csv("data/external/nusax/train.csv")
rows.append(pd.DataFrame({"indonesian": d["indonesian"], "javanese": d["javanese"], "source": "nusax", "peran": "umum"}))
x = pd.concat(rows, ignore_index=True).dropna()
buang = {}


def saring(mask, alasan):
    global x
    buang[alasan] = {k: int(v) for k, v in x[~mask].groupby("source").size().items()}
    x = x[mask].reset_index(drop=True)


for c in ["indonesian", "javanese"]:
    x[c] = x[c].map(norm)
lap["awal"] = x.groupby(["source", "peran"]).size().rename(lambda t: t, axis=0).to_dict()
lap["awal"] = {"/".join(k): int(v) for k, v in lap["awal"].items()}
ni, nj = x.indonesian.str.split().str.len(), x.javanese.str.split().str.len()
saring(ni.between(MIN_KATA, MAX_KATA) & nj.between(MIN_KATA, MAX_KATA) & (ni / nj).between(0.6, 1.6), "panjang")
noise = r"https?:|www\.|\[url\]|\[username\]|@\w|#\w|[^\x00-\x7FéèêÉÈ–—…]"
saring(~(x.indonesian.str.contains(noise, regex=True, case=False) | x.javanese.str.contains(noise, regex=True, case=False)), "url_mention_emoji")
saring(x.indonesian.map(kunci) != x.javanese.map(kunci), "sama_dengan_sumber")

ev = pd.concat([va, te])
tolak = np.zeros(len(x), bool)
for c in ["indonesian", "javanese"]:
    v = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 4)).fit(pd.concat([ev[c], x[c]]))
    sim = np.asarray((v.transform(x[c]) @ v.transform(ev[c]).T).max(1).todense()).ravel()
    lap[f"kemiripan_maks_{c}"] = round(float(sim.max()), 3)
    tolak |= sim >= MIRIP
saring(pd.Series(~tolak), "mirip_valid_test")

P = clf.predict_proba(x["javanese"])
x["level"], x["p_level"] = clf.classes_[P.argmax(1)], P.max(1).round(3)
boleh = {"ngoko": ["ngoko"], "krama": ["krama", "krama_alus"], "umum": ["ngoko"]}
saring(pd.Series([l in boleh[p] for l, p in zip(x.level, x.peran)]), "tingkat_tidak_cocok")
saring(x.p_level >= x.level.map(P_MIN), "label_tidak_yakin")

uu = tr[tr.level.isin(LEVELS)].assign(source="unggah_ungguh", p_level=1.0)
sudah = set(zip(uu.indonesian.map(kunci), uu.javanese.map(kunci)))
x["k"] = list(zip(x.indonesian.map(kunci), x.javanese.map(kunci)))
saring(~x.k.duplicated() & ~x.k.isin(sudah), "duplikat")
nt = x[x.source == "nusatranslation"]
if len(nt) > CAP_NUSATRANS:
    saring(~x.index.isin(nt.drop(nt.sample(CAP_NUSATRANS, random_state=SEED).index).index), "cap_nusatranslation")

x["group"] = 100000 + pd.factorize(x.indonesian.map(kunci))[0]
kol = ["group", "level", "indonesian", "javanese", "source", "p_level"]
out = pd.concat([uu[kol], x[kol]], ignore_index=True)
out.to_csv("data/merged/train_eksternal_3lv.csv", index=False, encoding="utf-8")
tab = out.groupby(["source", "level"]).size().unstack(fill_value=0)
for n, e in [("valid", va), ("test", te)]:
    lap[f"bocor_persis_{n}"] = {c: len(set(x[c].map(kunci)) & set(e[c].map(kunci))) for c in ["indonesian", "javanese"]}
lap.update(dibuang=buang, hasil={s: r.to_dict() for s, r in tab.iterrows()}, total={k: int(v) for k, v in tab.sum().items()})
json.dump(lap, open("data/merged/train_eksternal_3lv_laporan.json", "w", encoding="utf-8"), indent=1, ensure_ascii=False, default=int)
print(json.dumps(lap, indent=1, ensure_ascii=False, default=int)); print(tab)
'''

import urllib.request
W = "/tmp/eksternal"
for d in ["data/unggah_ungguh/split", "data/external/ngokokrama", "data/external/nusatranslation", "data/external/nusax", "data/merged"]:
    os.makedirs(f"{W}/{d}", exist_ok=True)
for n in ["train", "valid", "test"]:
    shutil.copy(find(n), f"{W}/data/unggah_ungguh/split/{n}.csv")


def unduh(u, p):
    for k in range(4):
        try:
            return urllib.request.urlretrieve(u, f"{W}/data/external/{p}")
        except Exception as e:
            log("unduh gagal", p, repr(e)); time.sleep(10)
    raise RuntimeError(p)


RAW = "https://raw.githubusercontent.com"
unduh(f"{RAW}/ir-nlp-csui/NgokoKrama/HEAD/dataset/NgokoKrama.xlsx", "ngokokrama/NgokoKrama.xlsx")
for s in ["train", "valid", "test"]:
    unduh(f"{RAW}/IndoNLP/nusa-writes/main/data/nusa_kalimat-mt-jav-{s}.csv", f"nusatranslation/jav-{s}.csv")
unduh(f"{RAW}/IndoNLP/nusax/main/datasets/mt/train.csv", "nusax/train.csv")
open(f"{W}/gabung.py", "w", encoding="utf-8").write(SKRIP)
p = subprocess.run([sys.executable, "gabung.py"], cwd=W, capture_output=True, text=True, env={**os.environ, "PYTHONUTF8": "1"})
print(p.stdout[-6000:], p.stderr[-3000:]); assert p.returncode == 0
EXT = pd.read_csv(f"{W}/data/merged/train_eksternal_3lv.csv")
MANIFEST["data_eksternal"] = json.load(open(f"{W}/data/merged/train_eksternal_3lv_laporan.json", encoding="utf-8"))
for k in ["bocor_persis_valid", "bocor_persis_test"]:
    assert all(v == 0 for v in MANIFEST["data_eksternal"][k].values()), k
EXT = EXT[EXT["source"] != "unggah_ungguh"].reset_index(drop=True)
shutil.rmtree(W)
log("eksternal", EXT.groupby(["source", "level"]).size().to_dict(), "| baris lokal 2192, didapat", len(EXT))


In [ ]:
# ---------------------------------------------------- MT (fungsi v2 persis)
def load():
    tok = AutoTokenizer.from_pretrained(REPO, src_lang=SL, tgt_lang=TL)
    model = AutoModelForSeq2SeqLM.from_pretrained(REPO)
    tok.add_tokens(list(TAG.values()), special_tokens=True)
    model.resize_token_embeddings(len(tok))
    emb = model.get_input_embeddings().weight
    with torch.no_grad():
        for l, words in INIT_WORDS.items():
            ids = tok(words, add_special_tokens=False)["input_ids"]
            emb[tok.convert_tokens_to_ids(TAG[l])] = emb[ids].mean(0)
    bos = tok.get_lang_id(TL)
    return tok, model.to(dev), bos


def translate(tok, model, bos, srcs, beams=4, bs=48):
    model.eval(); out = [None] * len(srcs)
    order = sorted(range(len(srcs)), key=lambda i: len(srcs[i]))
    for i in range(0, len(order), bs):
        idx = order[i:i + bs]
        enc = tok([srcs[j] for j in idx], return_tensors="pt", padding=True, truncation=True, max_length=MAXLEN).to(dev)
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            g = model.generate(**enc, forced_bos_token_id=bos, num_beams=beams, max_new_tokens=MAXLEN)
        for j, t in zip(idx, tok.batch_decode(g, skip_special_tokens=True)):
            out[j] = t
    return out


def seq_scores(tok, model, srcs, cands, chunk=16):
    out = []
    for i in range(0, len(srcs), chunk):
        enc = tok(srcs[i:i + chunk], text_target=cands[i:i + chunk], return_tensors="pt",
                  padding=True, truncation=True, max_length=MAXLEN).to(dev)
        labels = enc.pop("labels"); labels[labels == tok.pad_token_id] = -100
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            logits = model(**enc, labels=labels).logits
        nll = F.cross_entropy(logits.float().transpose(1, 2), labels, ignore_index=-100, reduction="none")
        m = (labels != -100).float()
        out += (-(nll * m).sum(1) / m.sum(1).clamp(min=1)).cpu().tolist()
        del logits, nll
    return np.array(out)


def nbest(tok, model, bos, srcs, k=NBEST, bs=8):
    model.eval(); out = [None] * len(srcs)
    order = sorted(range(len(srcs)), key=lambda i: len(srcs[i]))
    for i in range(0, len(order), bs):
        idx = order[i:i + bs]
        enc = tok([srcs[j] for j in idx], return_tensors="pt", padding=True, truncation=True, max_length=MAXLEN).to(dev)
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            g = model.generate(**enc, forced_bos_token_id=bos, num_beams=k, num_return_sequences=k, max_new_tokens=MAXLEN)
        txt = tok.batch_decode(g, skip_special_tokens=True)
        sc = seq_scores(tok, model, [srcs[j] for j in idx for _ in range(k)], txt)
        for n, j in enumerate(idx):
            out[j] = (txt[n * k:(n + 1) * k], sc[n * k:(n + 1) * k])
    torch.cuda.empty_cache()
    return out


def rerank(nb, levels, lam=LAM):
    flat = [t for c, _ in nb for t in c]
    lp = rr_lp(flat); res = []; p = 0
    for (c, s), l in zip(nb, levels):
        k = len(c); tot = s + lam * lp[p:p + k, LID[l]]; p += k
        res.append(c[int(np.argmax(tot))])
    return res


def ctrl_acc(hyps, levels):
    p = np.array(classify(hyps)); y = np.array([LID[l] for l in levels])
    return float((p == y).mean()), {l: round(float((p[y == i] == i).mean()), 4) for i, l in enumerate(LEVELS)}


def valid_score(h):
    c = chrf(h, va["javanese"].tolist()); a, alv = ctrl_acc(h, va["level"].tolist())
    return c + CTRL_W * a, round(c, 2), round(a, 4), alv


def per_level(h, d):
    res = {}
    for lv in LEVELS + ["all"]:
        m = (d["level"] == lv) if lv != "all" else d["level"].notna()
        hh, r = [x for x, k in zip(h, m) if k], d.loc[m, "javanese"].tolist()
        res[lv] = {"n": len(r), "chrF++": round(chrf(hh, r), 2), "BLEU": round(bleu(hh, r), 2)}
    return res


def honor_pairs(raw):
    pairs = []
    for _, g in raw.groupby("group"):
        jv = dict(zip(g["level"], g["javanese"]))
        if "krama" in jv and "krama_alus" in jv and jv["krama"] != jv["krama_alus"]:
            pairs.append((f"{TAG['krama_alus']} {jv['krama']}", jv["krama_alus"]))
    return pairs

In [ ]:
# ---------------------------------------------------- training satu lengan, nilai di VALID
def train_run(name, tr_sub, raw_sub, extra, seed=TRAIN_SEED):
    seed_all(seed)
    tok, model, bos = load()
    base = [(tag_src(s, l), t, l) for s, l, t in zip(tr_sub["indonesian"], tr_sub["level"], tr_sub["javanese"])]
    id_pairs = [(s, t) for s, t, _ in base] + [(s, t) for s, t, l in base if l == "krama_alus"] * (OS - 1)
    id_pairs += [(tag_src(s, l), t) for s, l, t in zip(extra["indonesian"], extra["level"], extra["javanese"])]
    jv_pairs = honor_pairs(raw_sub)

    def make_batches():
        a, b = id_pairs[:], jv_pairs[:]
        random.shuffle(a); random.shuffle(b)
        bt = [(SL, a[i:i + BS]) for i in range(0, len(a), BS)] + [(TL, b[i:i + BS]) for i in range(0, len(b), BS)]
        random.shuffle(bt); return bt

    n_batches = len(make_batches())
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = EPOCHS * n_batches
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP * steps), steps)
    scaler = torch.amp.GradScaler("cuda")
    best, best_state, hist = -1, None, []
    va_src = [tag_src(s, l) for s, l in zip(va["indonesian"], va["level"])]
    step = n_upd = ep = 0
    while step < steps:
        model.train(); tot = n = 0
        for src_lang, b in make_batches():
            if step >= steps:
                break
            tok.src_lang = src_lang
            enc = tok([s for s, _ in b], text_target=[t for _, t in b], return_tensors="pt",
                      padding=True, truncation=True, max_length=MAXLEN).to(dev)
            labels = enc.pop("labels"); labels[labels == tok.pad_token_id] = -100
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(**enc, labels=labels).logits
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), labels.view(-1),
                                   ignore_index=-100, label_smoothing=0.0)
            opt.zero_grad(); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            s0 = scaler.get_scale(); scaler.step(opt); scaler.update()
            n_upd += int(scaler.get_scale() >= s0); sched.step(); step += 1
            tot += loss.item() * len(b); n += len(b)
        ep += 1
        tok.src_lang = SL
        sc, c, a, alv = valid_score(translate(tok, model, bos, va_src, 1))
        hist.append({"epoch": ep, "step": step, "train_loss": round(tot / n, 4), "valid_chrF++": c, "valid_ctrl": a,
                     "valid_ctrl_lv": alv, "score": round(sc, 2)})
        log(name, hist[-1])
        if sc > best:
            best = sc; best_state = {k: x.detach().to("cpu", copy=True) for k, x in model.state_dict().items()}
    tok.src_lang = SL
    del opt, scaler, sched; gc.collect(); torch.cuda.empty_cache()
    out = {}
    for tag in ["pilih"]:
        if tag == "pilih":
            model.load_state_dict(best_state); del best_state; gc.collect()
        hyp = rerank(nbest(tok, model, bos, va_src), va["level"].tolist())
        sc, c, a, alv = valid_score(hyp)
        pv = va.copy(); pv["hyp"] = hyp; pv["juri_kernel"] = [LEVELS[q] for q in classify(hyp)]
        pv.to_csv(f"{OUT}/valid_{name}_{tag}.csv", index=False)
        out[tag] = {                   "valid_rerank": {"score": round(sc, 2), "chrF++": c, "ctrl": a, "ctrl_lv": alv, "per_level": per_level(hyp, va)},
                   "valid_csv_sha256": sha(f"{OUT}/valid_{name}_{tag}.csv")}
    res = {"n_pairs": [len(id_pairs), len(jv_pairs)], "n_batches_per_epoch": n_batches, "steps_scheduled": steps,
           "steps_done": step, "optimizer_updates": n_upd, "warmup": int(WARMUP * steps), "history": hist,
           "best_epoch": max(hist, key=lambda h: h["score"])["epoch"], **out}
    if name != "base":
        model.half().save_pretrained(f"{OUT}/model_{name}"); tok.save_pretrained(f"{OUT}/model_{name}")
    del model; gc.collect(); torch.cuda.empty_cache()
    log(f"{name} VALID", res["steps_done"], res["optimizer_updates"], res["pilih"]["valid_rerank"])
    return res




In [ ]:
# ---------------------------------------------------- lengan data
LENGAN = {"base": [], "ngokokrama": ["ngokokrama"], "tanpa_ngokokrama": ["nusax", "nusatranslation"],
          "semua": ["ngokokrama", "nusax", "nusatranslation"]}
V2_F100 = [31.94, 44.32, 51.49, 54.33, 57.35, 57.59, 59.6, 59.83, 60.03, 61.03, 61.1, 61.69, 61.62, 61.73, 61.77]
simpan = lambda: json.dump(MANIFEST, open(f"{OUT}/manifest.json", "w"), indent=1, default=str)
for name, srcs in LENGAN.items():
    ex = EXT[EXT["source"].isin(srcs)].reset_index(drop=True)
    t = time.time()
    try:
        r = train_run(name, tr, tr_raw, ex)
        r.update({"sumber": srcs, "n_tambahan": len(ex), "tambahan_per_level": {k: int(v) for k, v in ex["level"].value_counts().items()},
                  "detik": round(time.time() - t)})
    except Exception as e:
        import traceback; traceback.print_exc()
        r = {"error": repr(e)}
        gc.collect(); torch.cuda.empty_cache()
    MANIFEST["runs"][name] = r
    if name == "base" and "history" in r:
        MANIFEST["cek_base_identik"] = [h["valid_chrF++"] for h in r["history"]] == V2_F100
        log("cek base identik", MANIFEST["cek_base_identik"])
    simpan()

ok = {k: v["pilih"]["valid_rerank"] for k, v in MANIFEST["runs"].items() if "pilih" in v}
ka = {k: v["per_level"]["krama_alus"]["chrF++"] for k, v in ok.items()}
layak = [k for k in ok if "base" in ok and ka[k] >= ka["base"] - 1.0]
MANIFEST["ringkasan"] = {k: {"score": v["score"], "ctrl": v["ctrl"], "chrF++": v["chrF++"],
                             "per_level": {l: x["chrF++"] for l, x in v["per_level"].items()}, "ctrl_lv": v["ctrl_lv"]} for k, v in ok.items()}
win = max(layak, key=lambda k: ok[k]["score"]) if layak else None
MANIFEST["pemenang"] = win
for k in LENGAN:
    if k != win:
        shutil.rmtree(f"{OUT}/model_{k}", ignore_errors=True)
MANIFEST["elapsed_s"] = round(time.time() - T0); simpan()
print(json.dumps(MANIFEST["ringkasan"], indent=1)); log("PEMENANG (valid)", win); log("SELESAI")
